# Project 2

## Configuration

In [1]:
import time
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, TimestampType
)
# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [3]:
# Cleanup
import shutil
shutil.rmtree("/tmp/chk-bronze", ignore_errors=True)
shutil.rmtree("/tmp/chk-silver", ignore_errors=True)
spark.sql("TRUNCATE TABLE lakehouse.taxi.silver_trips")
spark.sql("TRUNCATE TABLE lakehouse.taxi.bronze_trips")
spark.sql("TRUNCATE TABLE lakehouse.taxi.gold_route_stats")
spark.sql("TRUNCATE TABLE lakehouse.taxi.streaming_control")
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.silver_trips")
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.gold_route_stats")


DataFrame[]

## Bronze layer

In [4]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze_trips (
    key STRING,
    value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    timestamp TIMESTAMP
)
USING iceberg
""")

bronze_source = (
    raw_stream
    .select(
        F.col("key").cast("string"),
        F.col("value").cast("string"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp")
    )
)

bronze_query = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)

count_before = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart: {count_before}")

bronze_query.stop()
print("Query stopped.")

# Restart the query using the SAME checkpoint directory.
# Even though startingOffsets is still 'earliest', Spark ignores that setting
# and resumes from the committed offsets in the checkpoint.

bronze_query2 = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)   # wait for two triggers

count_after = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart : {count_before}")
print(f"Row count after restart  : {count_after}")
print()
if count_after == count_before:
    print("Counts are equal — the checkpoint prevented re-ingestion of already-processed messages.")
else:
    print(f"Counts differ by {count_after - count_before} — "
          "those are new messages produced between the two runs.")

bronze_query2.stop()

spark.sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT value) AS unique_events
FROM lakehouse.taxi.bronze_trips;""").show()

Row count before restart: 4595
Query stopped.
Row count before restart : 4595
Row count after restart  : 4595

Counts are equal — the checkpoint prevented re-ingestion of already-processed messages.
+----------+-------------+
|total_rows|unique_events|
+----------+-------------+
|      4595|         2935|
+----------+-------------+



In [5]:
spark.sql("""
SELECT count(*) FROM lakehouse.taxi.bronze_trips;
""").show()

spark.sql("""
SELECT * FROM lakehouse.taxi.bronze_trips LIMIT 3;
""").show()
# The checkpoint location is configured in the bronze writeStream query: 
#.option("checkpointLocation", "/tmp/chk-bronze")

+--------+
|count(1)|
+--------+
|    4595|
+--------+

+---+--------------------+----------+---------+------+--------------------+
|key|               value|     topic|partition|offset|           timestamp|
+---+--------------------+----------+---------+------+--------------------+
|  1|{"VendorID": 1, "...|taxi-trips|        0|     0|2026-04-03 21:15:...|
|  1|{"VendorID": 1, "...|taxi-trips|        0|     1|2026-04-03 21:15:...|
|  1|{"VendorID": 1, "...|taxi-trips|        0|     2|2026-04-03 21:15:...|
+---+--------------------+----------+---------+------+--------------------+



# Silver layer

In [6]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.streaming_control (
    pipeline_name STRING,
    last_snapshot_id BIGINT
)
USING ICEBERG
""")

DataFrame[]

In [7]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.silver_trips (
    VendorID             INT,
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    passenger_count      INT,
    trip_distance        DOUBLE,
    PULocationID         INT,
    DOLocationID         INT,
    fare_amount          DOUBLE,
    total_amount         DOUBLE,
    pickup_borough       STRING,
    pickup_zone          STRING,
    pickup_service_zone  STRING,
    dropoff_borough      STRING,
    dropoff_zone         STRING,
    dropoff_service_zone STRING,
    kafka_timestamp      TIMESTAMP
)
USING ICEBERG
PARTITIONED BY (days(tpep_pickup_datetime))
""")

schema = StructType([
    StructField("VendorID",              IntegerType()),
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("total_amount",          DoubleType()),
])

pu = zones.alias("pu")
do = zones.alias("do")

In [8]:
SOURCE_TABLE = "lakehouse.taxi.bronze_trips"
TARGET_TABLE = "lakehouse.taxi.silver_trips"
GOLD_TABLE = "lakehouse.taxi.gold_route_stats"
CONTROL_TABLE = "lakehouse.taxi.streaming_control"
PIPELINE_ID = "silver_to_gold_stats"


In [9]:
def run_micro_batch():
    control_df = spark.table(CONTROL_TABLE).filter(F.col("pipeline_name") == PIPELINE_ID)
    last_id = control_df.select("last_snapshot_id").first()[0] if not control_df.isEmpty() else None

    try:
        latest_snap_df = spark.read.format("iceberg").load(f"{SOURCE_TABLE}.snapshots")
        current_snapshot_id = latest_snap_df.sort(F.desc("committed_at")).select("snapshot_id").first()[0]
    except Exception:
        print("Bronze table is empty or unavailable.")
        return

    if last_id == current_snapshot_id:
        print("No new data found. Skipping...")
        return

    if last_id is None:
        print("First run: Processing all historical data...")
        bronze_batch = spark.read.table(SOURCE_TABLE)
    else:
        print(f"Incremental run: Processing from {last_id} to {current_snapshot_id}")
        bronze_batch = (
            spark.read.format("iceberg")
            .option("start-snapshot-id", last_id)
            .load(SOURCE_TABLE)
        )

    parsed = (
        bronze_batch
        .withColumn("json", F.from_json(F.col("value").cast("string"), schema))
        .select("json.*", F.col("timestamp").alias("kafka_timestamp"))
    )

    deduplicated = parsed.dropDuplicates(["VendorID", "tpep_pickup_datetime", "PULocationID"])
    
    transformed = (
        deduplicated
        .select(
            F.col("kafka_timestamp"),
            F.col("VendorID").cast("int"),
            F.col("tpep_pickup_datetime"),
            F.col("tpep_dropoff_datetime"),
            F.when(
                (F.col("passenger_count").isNull()) | (F.col("passenger_count") <= 0), 1
            ).otherwise(F.col("passenger_count")).cast("int").alias("passenger_count"),
            F.col("trip_distance").cast("double"),
            F.col("PULocationID").cast("int"),
            F.col("DOLocationID").cast("int"),
            F.col("fare_amount").cast("double"),
            F.col("total_amount").cast("double"),
        )
        .dropna(subset=[
            "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "PULocationID", "DOLocationID"
        ])
        .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
        .filter(F.col("trip_distance") > 0)
        .filter((F.col("passenger_count") > 0) & (F.col("passenger_count") < 10))
        .filter(F.col("fare_amount") >= 0)
    )
    
    enriched = (
        transformed
        .join(F.broadcast(pu), transformed.PULocationID == F.col("pu.LocationID"), "left")
        .join(F.broadcast(do), transformed.DOLocationID == F.col("do.LocationID"), "left")
        .select(
            transformed["*"],
            F.col("pu.Borough").alias("pickup_borough"),
            F.col("pu.Zone").alias("pickup_zone"),
            F.col("pu.service_zone").alias("pickup_service_zone"),
            F.col("do.Borough").alias("dropoff_borough"),
            F.col("do.Zone").alias("dropoff_zone"),
            F.col("do.service_zone").alias("dropoff_service_zone"),
        )
        .filter(
            F.col("pickup_zone").isNotNull() &
            F.col("dropoff_zone").isNotNull()
        )
    )

    new_records_count = enriched.count()

    if new_records_count > 0:
        try:
            # Write to Silver
            enriched.writeTo(TARGET_TABLE).append()

            # Update Checkpoint
            spark.sql(f"""
                MERGE INTO {CONTROL_TABLE} t
                USING (SELECT '{PIPELINE_ID}' as id, {current_snapshot_id} as snap) s
                ON t.pipeline_name = s.id
                WHEN MATCHED THEN UPDATE SET last_snapshot_id = s.snap
                WHEN NOT MATCHED THEN INSERT (pipeline_name, last_snapshot_id) VALUES (s.id, s.snap)
            """)
            
            total_silver_rows = spark.read.table(TARGET_TABLE).count()
            
            print(f"New rows processed: {new_records_count}")
            print(f"Total rows in Silver: {total_silver_rows}")
            print(f"Current Snapshot ID: {current_snapshot_id}")

        except Exception as e:
            print(f"Batch failed: {e}")
    else:
        print("Batch produced 0 valid records after filtering. Skipping write.")

For simplicity sake, we run the micobatch job 5 times - in reality, it would keep running on the background and continuously ingest all incoming data

In [10]:
print("Starting micro-batch loop.")
for i in range (5):
    start_time = time.time()
    
    run_micro_batch()
    
    elapsed = time.time() - start_time
    sleep_time = max(0, 5 - elapsed)
    
    time.sleep(sleep_time)


Starting micro-batch loop.
First run: Processing all historical data...
New rows processed: 2773
Total rows in Silver: 2773
Current Snapshot ID: 5626123772476405526
No new data found. Skipping...
No new data found. Skipping...
No new data found. Skipping...
No new data found. Skipping...


In [11]:
silver_check = spark.read.table("lakehouse.taxi.silver_trips")
silver_check.show(5, truncate=False)
print(f"Silver count: {silver_check.count()}\n")

print("Null counts per column")
silver_check.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_check.columns
]).show()

print("Trip distribution by borough")
silver_check.groupBy("pickup_borough").count().show()

print("Trip duration statistics")
silver_check.select(
    (F.col("tpep_dropoff_datetime").cast("long") - 
     F.col("tpep_pickup_datetime").cast("long")).alias("duration_sec")
).describe().show()

print("Total vs unique")
spark.sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT *) AS unique_rows
FROM lakehouse.taxi.silver_trips;""").show()

+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------------------------+-------------------+---------------+-----------------+--------------------+-----------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|fare_amount|total_amount|pickup_borough|pickup_zone                  |pickup_service_zone|dropoff_borough|dropoff_zone     |dropoff_service_zone|kafka_timestamp        |
+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------------------------+-------------------+---------------+-----------------+--------------------+-----------------------+
|2       |2024-12-31 23:56:19 |2025-01-01 00:11:19  |1              |2.28         |68          |107         |14.9       |23.88       |Manhattan     |East Chelsea  

# Gold layer

In [12]:

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold_route_stats (
    pickup_borough    STRING,
    dropoff_borough   STRING,
    trip_count        BIGINT,
    total_revenue     DOUBLE,
    avg_distance      DOUBLE,
    revenue_per_mile  DOUBLE
)
USING iceberg
PARTITIONED BY (pickup_borough)
""")

DataFrame[]

In [13]:
def update_gold_route_stats():
    """
    Reads from Silver, calculates aggregates, and overwrites the Gold table.
    """

    gold_route_df = (
        spark.table("lakehouse.taxi.silver_trips")
        .groupBy("pickup_borough", "dropoff_borough")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.sum("total_amount"), 2).alias("total_revenue"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
            F.round(
                F.sum("total_amount") / F.when(F.sum("trip_distance") > 0, F.sum("trip_distance")).otherwise(1), 
                2
            ).alias("revenue_per_mile")
        )
    )

    gold_route_df.writeTo("lakehouse.taxi.gold_route_stats").overwritePartitions()

#### Verify, that restart does not produce duplicates

In [14]:
# initial run
update_gold_route_stats()
count_initial = spark.table("lakehouse.taxi.gold_route_stats").count()
print(f"Row count after initial run: {count_initial}")
print(f"Data after initial run: {count_initial}")
spark.table("lakehouse.taxi.gold_route_stats").sort(F.desc("trip_count")).show()

# duplicate run
update_gold_route_stats()
count_after_restart = spark.table("lakehouse.taxi.gold_route_stats").count()


print(f"Row count second run  : {count_after_restart}")
print(f"Data after second run: {count_initial}")
spark.table("lakehouse.taxi.gold_route_stats").sort(F.desc("trip_count")).show()

Row count after initial run: 17
Data after initial run: 17
+--------------+---------------+----------+-------------+------------+----------------+
|pickup_borough|dropoff_borough|trip_count|total_revenue|avg_distance|revenue_per_mile|
+--------------+---------------+----------+-------------+------------+----------------+
|     Manhattan|      Manhattan|      2490|     56044.51|        2.11|           10.65|
|     Manhattan|       Brooklyn|        90|      4007.57|        6.32|            7.04|
|     Manhattan|         Queens|        46|       2071.8|        7.34|            6.13|
|        Queens|      Manhattan|        31|      2124.33|        11.2|            6.12|
|        Queens|       Brooklyn|        28|      1706.79|       11.48|            5.31|
|        Queens|         Queens|        20|       678.68|         5.5|            6.17|
|     Manhattan|          Bronx|        18|       889.04|        9.33|            5.29|
|      Brooklyn|      Manhattan|         9|       434.74|    

In [15]:
gold_audit_df = spark.sql("""
    SELECT 
        snapshot_id,
        committed_at,
        operation,
        summary['replace-partitions'] AS is_overwrite,
        summary['added-records']      AS rows_added,
        summary['deleted-records']    AS rows_deleted,
        summary['total-records']      AS total_table_rows,
        summary['added-data-files']   AS files_added,
        summary['deleted-data-files'] AS files_deleted
    FROM lakehouse.taxi.gold_route_stats.snapshots
    ORDER BY committed_at DESC
""")

gold_audit_df.show(truncate=False)

+-------------------+-----------------------+---------+------------+----------+------------+----------------+-----------+-------------+
|snapshot_id        |committed_at           |operation|is_overwrite|rows_added|rows_deleted|total_table_rows|files_added|files_deleted|
+-------------------+-----------------------+---------+------------+----------+------------+----------------+-----------+-------------+
|1924628277674339238|2026-04-04 09:14:45.099|overwrite|true        |17        |17          |17              |4          |4            |
|7454032751276084582|2026-04-04 09:14:44.152|overwrite|true        |17        |NULL        |17              |4          |NULL         |
+-------------------+-----------------------+---------+------------+----------+------------+----------------+-----------+-------------+

